# Hyde and Hand: a reproducible computational case study

This notebook reconstructs selected computational questions from Laura Elizabeth Hand's literary research on Robert Louis Stevenson's *The Strange Case of Dr. Jekyll and Mr. Hyde*. It does **not** attempt to prove a literary interpretation from counts. Instead, it documents a research workflow in which quantitative patterns identify passages, distributions, and structural relationships that can be returned to close reading.

The underlying argument treats the hand in three interconnected roles: as a mediator of homosocial relation and intimacy; as a bodily marker of Jekyll/Hyde transformation; and as the material mechanism of writing, handwriting, letters, confession, and readerly transmission.

## 1. Reproducible corpus and research vocabulary

The text is retrieved from the public-domain Project Gutenberg source recorded in the repository manifest. Research terms and phrases are version-controlled separately from the notebook.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from stevenson_text.analysis import concordance, count_terms, segment_term_counts, significant_collocates
from stevenson_text.corpus import fetch_document
from stevenson_text.phrases import phrase_counts, phrase_occurrences
from stevenson_text.statistics import dispersion_profile, log_likelihood_keyness
from stevenson_text.structure import section_entity_matrix, split_by_headings

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
manifest = pd.read_csv(ROOT / 'data/source_manifest.csv')
research = json.loads((ROOT / 'data/research_terms.json').read_text(encoding='utf-8'))
aliases = json.loads((ROOT / 'data/character_aliases.json').read_text(encoding='utf-8'))
structure = json.loads((ROOT / 'data/structure_rules.json').read_text(encoding='utf-8'))

jekyll_row = manifest.loc[manifest['ebook_id'] == 43].iloc[0]
jekyll = fetch_document(jekyll_row['title'], jekyll_row['source_url'])
len(jekyll.tokens)


## 2. Presence and absence: gendered vocabulary

One observation in the original research was the striking imbalance between male and female terms. A raw token ratio is not a theory of gender; it is a prompt to ask how the novel distributes social presence.

In [ ]:
gender_terms = ['man', 'men', 'woman', 'women', 'gentleman', 'friend']
pd.Series(count_terms(jekyll.tokens, gender_terms), name='count').to_frame()


### Interpretive checkpoint

A count should be read against narrative function and grammar. The useful result is the asymmetry itself, which motivates closer inspection of where gendered terms occur and whom they describe.

## 3. The hand as a distributed motif

The argument depends not merely on the frequency of `hand`, but on its recurrence across different narrative functions. Dispersion helps distinguish a motif distributed throughout the novel from one concentrated in a single episode.

In [ ]:
motifs = ['hand', 'hands', 'strange', 'peculiar', 'letter', 'writing', 'written', 'signature', 'confession']
pd.DataFrame([dispersion_profile(jekyll.tokens, term, segments=10) for term in motifs])


In [ ]:
trajectory = pd.DataFrame(segment_term_counts(jekyll.tokens, ['hand', 'strange', 'letter'], segments=10))
trajectory


The equal-length segments above are useful for comparison, but they are not chapters. Later sections repeat the analysis against authored narrative divisions.

## 4. Returning from count to context: concordance

The original essay distinguishes clasping hands, transformed bodies, handwriting, letters passing from hand to hand, and the reader's implied hand turning pages. A concordance lets us inspect whether repeated tokens actually participate in those categories.

In [ ]:
hand_kwic = pd.DataFrame(concordance(jekyll.text, 'hand', window=12, max_hits=None))
hand_kwic.head(25)


A researcher can code these contexts by function—for example **bodily transformation**, **touch/intimacy**, **handwriting/signature**, **document transfer**, or **idiom**. That annotation layer is a natural next step for TEI/XML.

## 5. Phrases central to the argument

Phrase-aware retrieval matters because a literary claim can turn on a construction rather than a unigram.

In [ ]:
phrases = ['written hand', 'very great interest', 'poor Hyde', 'my own hand', 'odd hand']
pd.Series(phrase_counts(jekyll.text, phrases), name='count').to_frame()


In [ ]:
pd.DataFrame(phrase_occurrences(jekyll.text, 'odd hand', window=18))


## 6. Collocation: what gathers around `hand`?

PMI can surface words occurring near `hand` more often than their corpus frequency alone would suggest. Raw co-occurrence and corpus counts remain visible.

In [ ]:
pd.DataFrame(significant_collocates(jekyll.tokens, 'hand', window=7, min_count=2)).head(30)


## 7. Authored structure and character relations

The essay argues that hands mediate relationships among Utterson, Jekyll, Hyde, Lanyon, Guest, Poole, and others. The repository can map canonical character aliases against the novel's named chapters.

In [ ]:
sections = split_by_headings(jekyll.text, structure[jekyll.title]['headings'])
character_matrix = pd.DataFrame(section_entity_matrix(sections, aliases[jekyll.title]))
character_matrix


## 8. Comparative corpus: is `hand` distinctive here?

Comparing *Jekyll and Hyde* with another Stevenson text provides a limited internal reference corpus. Log-likelihood and log ratio can identify relative over- and under-use while remaining descriptive.

In [ ]:
treasure_row = manifest.loc[manifest['ebook_id'] == 120].iloc[0]
treasure = fetch_document(treasure_row['title'], treasure_row['source_url'])
keyness = pd.DataFrame(log_likelihood_keyness(jekyll.tokens, treasure.tokens, min_total=5))
keyness.loc[keyness['term'].isin(['hand','hands','strange','letter','man','woman','face','voice'])]


A two-text comparison cannot establish period-wide distinctiveness. It functions as a falsification step: if a supposedly distinctive motif is equally or more characteristic of the comparison text, the literary claim needs refinement.

## 9. From computation back to close reading

The most consequential scenes in the original argument are not reducible to frequency. They involve changes in grammar, agency, materiality, and circulation: the transformed body marked first by **the hand**; the possibility of retaining identity because Jekyll can still **write [his] own hand**; the **odd hand** of the letter scrutinized by readers inside the novel; and documents literally passed into another man's hands.

The computational workflow contributes by showing where these motifs recur, how widely they are distributed, which terms cluster around them, which characters dominate the surrounding structures, and where exact phrases appear. Close reading remains necessary to interpret pronouns, puns, bodily agency, epistolary framing, and the historical meanings of strangeness, confession, intimacy, and handwriting.

## 10. Research extension

The original essay closes by proposing a broader quantitative project across Stevenson's peers and the Victorian canon. This repository now provides infrastructure for that extension: reproducible corpus provenance, configurable lexical/phrase analysis, authored structure, entity maps, TEI/XML, correspondence networks, and auditable browser outputs.